# ViT-Tiny Malware Classifier — Simplified Stage-by-Stage Implementation

This notebook **re-implements the full `vit_tiny_patch16_224` architecture from scratch**, loading the  
exact trained weights from `vit_malware.pth`.  Every stage is written as a standalone, inspectable Python  
class so the data-flow can be verified numerically at each step — the first checkpoint towards RTL.

---
### Architecture at a glance
| Stage | Operation | I/O shapes |
|---|---|---|
| **0** | Input image | `(1, 3, 224, 224)` |
| **1** | Patch embedding (Conv2d 16×16, stride 16) | → `(1, 196, 192)` |
| **2** | Prepend CLS token | → `(1, 197, 192)` |
| **3** | Add positional embedding | → `(1, 197, 192)` |
| **4** | Dropout (no-op at inference) | → `(1, 197, 192)` |
| **5** | 12 × Transformer Block | → `(1, 197, 192)` each |
| — | └─ LayerNorm 1 | → `(1, 197, 192)` |
| — | └─ Multi-Head Self-Attention (3 heads, head_dim=64) | → `(1, 197, 192)` |
| — | └─ Residual add | → `(1, 197, 192)` |
| — | └─ LayerNorm 2 | → `(1, 197, 192)` |
| — | └─ MLP: Linear(192→768) → GELU → Linear(768→192) | → `(1, 197, 192)` |
| — | └─ Residual add | → `(1, 197, 192)` |
| **6** | Final LayerNorm | → `(1, 197, 192)` |
| **7** | Extract CLS token | → `(1, 192)` |
| **8** | Classification head Linear(192→2) | → `(1, 2)` |
| **9** | Softmax → predicted class | — |

**Hyperparameters:** embed_dim=192, depth=12, num_heads=3, head_dim=64, mlp_hidden=768, patch_size=16, img_size=224

---
## 0 — Imports & constants

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import os

# ─── Model hyper-parameters (vit_tiny_patch16_224) ───
IMG_SIZE    = 224
PATCH_SIZE  = 16
IN_CHANS    = 3
EMBED_DIM   = 192
DEPTH       = 12        # number of transformer blocks
NUM_HEADS   = 3
HEAD_DIM    = EMBED_DIM // NUM_HEADS   # 64
MLP_RATIO   = 4
MLP_HIDDEN  = EMBED_DIM * MLP_RATIO   # 768
NUM_CLASSES = 2
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2   # 196
NUM_TOKENS  = NUM_PATCHES + 1                  # 197 (+ CLS)
ATTN_SCALE  = 1.0 / math.sqrt(HEAD_DIM)       # 1/sqrt(64) = 0.125

print("ViT-Tiny configuration")
print(f"  Image size    : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Patch size    : {PATCH_SIZE}x{PATCH_SIZE}")
print(f"  Num patches   : {NUM_PATCHES}")
print(f"  Sequence len  : {NUM_TOKENS} (patches + CLS)")
print(f"  Embed dim     : {EMBED_DIM}")
print(f"  Depth         : {DEPTH} blocks")
print(f"  Heads         : {NUM_HEADS}  (head_dim={HEAD_DIM})")
print(f"  MLP hidden    : {MLP_HIDDEN}")
print(f"  Attn scale    : {ATTN_SCALE}")
print(f"  Num classes   : {NUM_CLASSES}")

---
## Stage 1 — Patch Embedding

A single `Conv2d(in=3, out=192, kernel=16, stride=16)` slides across the 224×224 image  
producing a 14×14 grid of 192-dimensional token embeddings, then flattened to sequence length 196.

**RTL view:** this is a bank of 192 independent 16×16×3 dot-product filters applied with stride 16.

In [ ]:
class SimplePatchEmbed(nn.Module):
    """
    Splits the image into non-overlapping 16x16 patches and linearly projects
    each patch to a 192-dimensional vector.

    Equivalent to Conv2d(3, 192, kernel_size=16, stride=16) + flatten.
    """
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(
            IN_CHANS, EMBED_DIM,
            kernel_size=PATCH_SIZE,
            stride=PATCH_SIZE,
            bias=True
        )
        # After conv: (B, 192, 14, 14)
        # After flatten+transpose: (B, 196, 192)

    def forward(self, x):
        # x: (B, 3, 224, 224)
        x = self.proj(x)            # (B, 192, 14, 14)
        B, C, H, W = x.shape
        x = x.flatten(2)            # (B, 192, 196)
        x = x.transpose(1, 2)       # (B, 196, 192)  ← sequence first
        return x


# ── Quick shape test ──
_pe = SimplePatchEmbed()
_dummy = torch.zeros(1, 3, 224, 224)
_out = _pe(_dummy)
print(f"PatchEmbed input  : {list(_dummy.shape)}")
print(f"PatchEmbed output : {list(_out.shape)}   ← expect (1, 196, 192)")

---
## Stage 2 & 3 — CLS Token + Positional Embedding

- A learnable `[CLS]` token `(1, 1, 192)` is prepended to the 196 patch tokens → sequence length 197.
- A learnable positional embedding table `(1, 197, 192)` is added element-wise.

**RTL view:** Positional embedding is a ROM lookup; CLS token is a register bank.

In [ ]:
_cls  = torch.zeros(1, 1, EMBED_DIM)       # (1, 1, 192)
_pos  = torch.zeros(1, NUM_TOKENS, EMBED_DIM)  # (1, 197, 192)

def prepend_cls_and_pos(tokens, cls_token, pos_embed):
    """
    tokens    : (B, 196, 192)  – patch embeddings
    cls_token : (1, 1,   192)  – learnable parameter
    pos_embed : (1, 197, 192)  – learnable parameter
    Returns   : (B, 197, 192)
    """
    B = tokens.shape[0]
    cls_expanded = cls_token.expand(B, -1, -1)     # (B, 1, 192)
    x = torch.cat([cls_expanded, tokens], dim=1)   # (B, 197, 192)
    x = x + pos_embed                              # (B, 197, 192) broadcast
    return x

_seq = prepend_cls_and_pos(_out, _cls, _pos)
print(f"After CLS + PosEmb: {list(_seq.shape)}   ← expect (1, 197, 192)")

---
## Stage 4 — LayerNorm

Used **twice** inside every transformer block (before attention and before MLP).
Normalises the last dimension (192) independently for each token.

$$\text{LN}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

**RTL view:** mean and variance computed over 192-element vector; multiply-add with learnable scale/shift.

In [ ]:
class SimpleLayerNorm(nn.Module):
    """
    Layer norm over the last dimension (embed_dim = 192).
    gamma and beta are learnable parameters of shape (embed_dim,).
    """
    def __init__(self, dim=EMBED_DIM, eps=1e-6):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(dim))   # scale
        self.beta  = nn.Parameter(torch.zeros(dim))  # shift

    def forward(self, x):
        # x: (..., 192)
        mean = x.mean(dim=-1, keepdim=True)                # (..., 1)
        var  = x.var(dim=-1, unbiased=False, keepdim=True) # (..., 1)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)   # (..., 192)
        return self.gamma * x_norm + self.beta             # (..., 192)


_ln = SimpleLayerNorm()
_test = torch.randn(1, 197, 192)
_lnout = _ln(_test)
print(f"LayerNorm input  : {list(_test.shape)}")
print(f"LayerNorm output : {list(_lnout.shape)}   ← expect (1, 197, 192)")
# Each token's output should have approximately mean~0, std~1 before gamma/beta
print(f"Per-token mean (first token): {_lnout[0,0].mean().item():.4f}  (≈0)")
print(f"Per-token std  (first token): {_lnout[0,0].std().item():.4f}   (≈1)")

---
## Stage 5 — Multi-Head Self-Attention (MHSA)

**3 heads, head_dim = 64, scale = 0.125**

Steps per head:
1. Linear project x → Q, K, V each `(B, heads, seq, head_dim)`
2. Scaled dot-product: `scores = Q @ Kᵀ * scale`  →  `(B, heads, seq, seq)`
3. Softmax over last dim (also called the attention map)
4. Context: `attn @ V`  →  `(B, heads, seq, head_dim)`
5. Merge heads → `(B, seq, embed_dim)`
6. Output projection Linear(192→192)

**RTL view:** Steps 1–4 are the core GEMM pipeline per head; step 5 is a permute/reshape; step 6 is another GEMM.

In [ ]:
class SimpleMHSA(nn.Module):
    """
    Multi-Head Self-Attention — transparent, step-by-step.

    num_heads = 3, head_dim = 64, embed_dim = 192
    """
    def __init__(self, num_heads=NUM_HEADS, qkv_bias=True):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = EMBED_DIM // num_heads   # 64
        self.scale     = 1.0 / math.sqrt(self.head_dim)

        # Single fused projection: Q, K, V are computed in one Linear call
        # Weight shape: (3*192, 192) = (576, 192)
        self.qkv  = nn.Linear(EMBED_DIM, 3 * EMBED_DIM, bias=qkv_bias)
        self.proj = nn.Linear(EMBED_DIM, EMBED_DIM)   # output projection

    def forward(self, x):
        B, N, C = x.shape   # (B, 197, 192)

        # ── Step 1: Project to Q, K, V ──────────────────────────────────────
        qkv = self.qkv(x)                # (B, 197, 576)
        # Reshape: (B, 197, 3, num_heads, head_dim)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim)
        # Permute: (3, B, num_heads, 197, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        Q, K, V = qkv.unbind(0)          # each (B, num_heads, 197, head_dim)

        # ── Step 2: Scaled dot-product attention scores ──────────────────────
        # (B, heads, 197, 64) @ (B, heads, 64, 197) = (B, heads, 197, 197)
        scores = (Q @ K.transpose(-2, -1)) * self.scale

        # ── Step 3: Softmax (row-wise over key dimension) ────────────────────
        attn = F.softmax(scores, dim=-1)   # (B, heads, 197, 197)

        # ── Step 4: Weighted sum of values ───────────────────────────────────
        # (B, heads, 197, 197) @ (B, heads, 197, 64) = (B, heads, 197, 64)
        context = attn @ V

        # ── Step 5: Merge heads ──────────────────────────────────────────────
        # (B, heads, 197, 64) → (B, 197, heads, 64) → (B, 197, 192)
        context = context.transpose(1, 2).reshape(B, N, C)

        # ── Step 6: Output projection ────────────────────────────────────────
        out = self.proj(context)           # (B, 197, 192)

        return out, attn   # return attention map for inspection


_mhsa = SimpleMHSA()
_x = torch.randn(1, 197, 192)
_out_attn, _attn_map = _mhsa(_x)
print(f"MHSA input        : {list(_x.shape)}")
print(f"MHSA output       : {list(_out_attn.shape)}   ← expect (1, 197, 192)")
print(f"Attention map     : {list(_attn_map.shape)}   ← (B, heads, seq, seq)")
print(f"Attn row-sum (≈1) : {_attn_map[0, 0, 0].sum().item():.6f}")

---
## Stage 6 — MLP Block (Feed-Forward Network)

Two linear layers with **GELU** activation between them:

$$\text{MLP}(x) = \text{Linear}_{768 \to 192}(\,\text{GELU}(\text{Linear}_{192 \to 768}(x))\,)$$

GELU: $\text{GELU}(x) = x \cdot \Phi(x)$ where $\Phi$ is the standard normal CDF.  
Approximation used in practice: $x \cdot 0.5\,(1 + \tanh(\sqrt{2/\pi}\,(x + 0.044715 x^3)))$

**RTL view:** Two GEMMs (systolic arrays) with a GELU LUT between them — this is the `W1.mem`/`W2.mem` target.

In [ ]:
class SimpleMLP(nn.Module):
    """
    MLP block: Linear(192→768) → GELU → Linear(768→192)
    """
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(EMBED_DIM, MLP_HIDDEN)  # 192 → 768
        self.fc2 = nn.Linear(MLP_HIDDEN, EMBED_DIM)  # 768 → 192

    def forward(self, x):
        # x: (B, 197, 192)
        x = self.fc1(x)        # (B, 197, 768)  ← GEMM1 (W1)
        x = F.gelu(x)          # (B, 197, 768)  ← GELU activation
        x = self.fc2(x)        # (B, 197, 192)  ← GEMM2 (W2)
        return x


_mlp = SimpleMLP()
_x = torch.randn(1, 197, 192)
_out_mlp = _mlp(_x)
print(f"MLP input  : {list(_x.shape)}")
print(f"MLP output : {list(_out_mlp.shape)}   ← expect (1, 197, 192)")

# ── GELU approximation demo (for a scalar) ──────────────────────────────────
print("\nGELU sanity (a few values):")
for v in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    gelu_val = F.gelu(torch.tensor(v)).item()
    print(f"  GELU({v:+.1f}) = {gelu_val:.4f}")

---
## Stage 7 — Full Transformer Block

One block = two sub-layers each with a **pre-norm residual** connection:

```
x  ←  x + MHSA(LN1(x))
x  ←  x + MLP(LN2(x))
```

This is repeated **12 times** in ViT-Tiny.

In [ ]:
class SimpleTransformerBlock(nn.Module):
    """
    One ViT encoder block:
        x = x + Attention(LayerNorm1(x))
        x = x + MLP(LayerNorm2(x))
    """
    def __init__(self):
        super().__init__()
        self.norm1 = SimpleLayerNorm()
        self.attn  = SimpleMHSA()
        self.norm2 = SimpleLayerNorm()
        self.mlp   = SimpleMLP()

    def forward(self, x):
        # x: (B, 197, 192)

        # ── Sub-layer 1: Attention ──────────────────────────────────────────
        residual = x
        x_norm   = self.norm1(x)                  # (B, 197, 192)
        attn_out, attn_map = self.attn(x_norm)    # (B, 197, 192)
        x = residual + attn_out                   # residual connection

        # ── Sub-layer 2: MLP ────────────────────────────────────────────────
        residual = x
        x_norm   = self.norm2(x)                  # (B, 197, 192)
        mlp_out  = self.mlp(x_norm)               # (B, 197, 192)
        x = residual + mlp_out                    # residual connection

        return x, attn_map


_blk = SimpleTransformerBlock()
_x = torch.randn(1, 197, 192)
_out_blk, _amap = _blk(_x)
print(f"Block input  : {list(_x.shape)}")
print(f"Block output : {list(_out_blk.shape)}   ← expect (1, 197, 192)")
print(f"Attn map     : {list(_amap.shape)}       ← (1, 3 heads, 197, 197)")

---
## Stage 8 — Classification Head

After all 12 transformer blocks:
1. Apply final `LayerNorm` to the full sequence
2. Extract only the **CLS token** (index 0, position in sequence)
3. Pass through `Linear(192 → 2)` to get class logits
4. `Softmax` → probability distribution over {benign, malware}

In [ ]:
class SimpleClassificationHead(nn.Module):
    """
    Final LayerNorm → CLS token extraction → Linear classifier
    """
    def __init__(self):
        super().__init__()
        self.norm = SimpleLayerNorm()
        self.head = nn.Linear(EMBED_DIM, NUM_CLASSES)

    def forward(self, x):
        # x: (B, 197, 192)
        x   = self.norm(x)          # (B, 197, 192) — final LayerNorm
        cls = x[:, 0]               # (B, 192)       — extract CLS token
        out = self.head(cls)        # (B, 2)         — logits
        return out


_head = SimpleClassificationHead()
_x = torch.randn(1, 197, 192)
_logits = _head(_x)
print(f"Head input  : {list(_x.shape)}")
print(f"Logits      : {list(_logits.shape)}   ← expect (1, 2)")
print(f"Softmax     : {F.softmax(_logits, dim=-1).squeeze().tolist()}")

---
## Stage 9 — Complete Simplified ViT Model

Assembles all stages above into a single `nn.Module` with identical forward logic  
to the original `timm` model.

In [ ]:
class SimpleViT(nn.Module):
    """
    Complete ViT-Tiny Malware Classifier.

    Exact functional equivalent of:
        model = timm.create_model('vit_tiny_patch16_224', pretrained=False)
        model.head = nn.Linear(192, 2)

    Written stage-by-stage for direct RTL translation.
    """
    def __init__(self):
        super().__init__()

        # ── Stage 1: Patch embedding ──────────────────────────────────────
        self.patch_embed = SimplePatchEmbed()

        # ── Stage 2: CLS token (learnable) ───────────────────────────────
        self.cls_token = nn.Parameter(torch.zeros(1, 1, EMBED_DIM))

        # ── Stage 3: Positional embedding (learnable, 197 positions) ─────
        self.pos_embed = nn.Parameter(torch.zeros(1, NUM_TOKENS, EMBED_DIM))

        # ── Stage 4: Input dropout (identity at inference) ─────────────
        self.pos_drop = nn.Dropout(p=0.0)

        # ── Stage 5: 12 × Transformer Block ──────────────────────────────
        self.blocks = nn.ModuleList([
            SimpleTransformerBlock() for _ in range(DEPTH)
        ])

        # ── Stage 6+: Final norm + CLS head ──────────────────────────────
        self.clf_head = SimpleClassificationHead()

    def forward(self, x, return_attn=False):
        """
        x           : (B, 3, 224, 224)
        return_attn : if True, also return list of attention maps per block
        """
        # Stage 1 ─ Patch embedding
        x = self.patch_embed(x)                        # (B, 196, 192)

        # Stage 2 ─ Prepend CLS token
        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1)         # (B, 1, 192)
        x   = torch.cat([cls, x], dim=1)               # (B, 197, 192)

        # Stage 3 ─ Add positional embedding
        x = x + self.pos_embed                         # (B, 197, 192)

        # Stage 4 ─ Dropout (no-op at inference)
        x = self.pos_drop(x)

        # Stage 5 ─ 12 transformer blocks
        attn_maps = []
        for blk in self.blocks:
            x, attn = blk(x)                           # (B, 197, 192)
            attn_maps.append(attn)

        # Stage 6 + head ─ LayerNorm → CLS → Linear
        logits = self.clf_head(x)                      # (B, 2)

        if return_attn:
            return logits, attn_maps
        return logits


_simple_model = SimpleViT()
_dummy = torch.zeros(1, 3, 224, 224)
with torch.no_grad():
    _logits_test = _simple_model(_dummy)
print(f"SimpleViT output : {list(_logits_test.shape)}   ← expect (1, 2) ✓")
print(f"Total parameters : {sum(p.numel() for p in _simple_model.parameters()):,}")

---
## Stage 10 — Load Trained Weights into SimpleViT

Map every parameter from the `timm`-trained `state_dict` into the corresponding  
parameter of our `SimpleViT`.  The key names differ, so we build an explicit mapping table.

In [ ]:
WEIGHTS_PATH = "vit_malware.pth"
assert os.path.exists(WEIGHTS_PATH), f"'{WEIGHTS_PATH}' not found."

# ── Load reference model (timm) to use as weight source ──
ref_model = timm.create_model('vit_tiny_patch16_224', pretrained=False)
ref_model.head = nn.Linear(ref_model.head.in_features, NUM_CLASSES)
ref_sd = torch.load(WEIGHTS_PATH, map_location="cpu")
ref_model.load_state_dict(ref_sd)
ref_model.eval()

simple_model = SimpleViT()
target_sd = simple_model.state_dict()

# ── Build key mapping: timm name → SimpleViT name ──────────────────────────
mapping = {
    # Patch embedding
    "patch_embed.proj.weight" : "patch_embed.proj.weight",
    "patch_embed.proj.bias"   : "patch_embed.proj.bias",

    # CLS token & positional embedding
    "cls_token"               : "cls_token",
    "pos_embed"               : "pos_embed",
}

# Per-block mappings (12 blocks)
for i in range(DEPTH):
    p = f"blocks.{i}"   # timm prefix
    s = f"blocks.{i}"   # simple prefix (same index, different sub-keys)

    sub = {
        # LayerNorm 1 (before attention)
        f"{p}.norm1.weight" : f"{s}.norm1.gamma",
        f"{p}.norm1.bias"   : f"{s}.norm1.beta",

        # Attention QKV
        f"{p}.attn.qkv.weight" : f"{s}.attn.qkv.weight",
        f"{p}.attn.qkv.bias"   : f"{s}.attn.qkv.bias",

        # Attention output projection
        f"{p}.attn.proj.weight" : f"{s}.attn.proj.weight",
        f"{p}.attn.proj.bias"   : f"{s}.attn.proj.bias",

        # LayerNorm 2 (before MLP)
        f"{p}.norm2.weight" : f"{s}.norm2.gamma",
        f"{p}.norm2.bias"   : f"{s}.norm2.beta",

        # MLP
        f"{p}.mlp.fc1.weight" : f"{s}.mlp.fc1.weight",
        f"{p}.mlp.fc1.bias"   : f"{s}.mlp.fc1.bias",
        f"{p}.mlp.fc2.weight" : f"{s}.mlp.fc2.weight",
        f"{p}.mlp.fc2.bias"   : f"{s}.mlp.fc2.bias",
    }
    mapping.update(sub)

# Final norm  (timm: 'norm', SimpleViT: 'clf_head.norm')
mapping["norm.weight"] = "clf_head.norm.gamma"
mapping["norm.bias"]   = "clf_head.norm.beta"

# Classification head
mapping["head.weight"] = "clf_head.head.weight"
mapping["head.bias"]   = "clf_head.head.bias"

# ── Transfer weights using the mapping ──────────────────────────────────────
ref_sd_full = ref_model.state_dict()
transferred, skipped = 0, []

for src_key, dst_key in mapping.items():
    if src_key in ref_sd_full and dst_key in target_sd:
        assert ref_sd_full[src_key].shape == target_sd[dst_key].shape, (
            f"Shape mismatch: {src_key} {ref_sd_full[src_key].shape} "
            f"vs {dst_key} {target_sd[dst_key].shape}"
        )
        target_sd[dst_key] = ref_sd_full[src_key].clone()
        transferred += 1
    else:
        skipped.append((src_key, dst_key))

simple_model.load_state_dict(target_sd)
simple_model.eval()

print(f"✅  Weight transfer complete")
print(f"   Transferred : {transferred} tensors")
if skipped:
    print(f"   ⚠️  Skipped   : {len(skipped)}")
    for s, d in skipped:
        print(f"      {s} → {d}")

---
## Stage 11 — Numerical Equivalence Verification

Run both the original `timm` model and our `SimpleViT` on the same random input  
and confirm the outputs are **bit-for-bit identical** (within floating-point tolerance).

In [ ]:
torch.manual_seed(42)
test_input = torch.randn(2, 3, 224, 224)  # batch of 2 images

with torch.no_grad():
    ref_out    = ref_model(test_input)       # timm model
    simple_out = simple_model(test_input)    # our implementation

max_abs_diff = (ref_out - simple_out).abs().max().item()
mean_abs_diff = (ref_out - simple_out).abs().mean().item()

print("── Equivalence check ──────────────────────────")
print(f"  ref_model output    : {ref_out.tolist()}")
print(f"  simple_model output : {simple_out.tolist()}")
print(f"  Max  |diff|         : {max_abs_diff:.2e}")
print(f"  Mean |diff|         : {mean_abs_diff:.2e}")

TOLERANCE = 1e-4
if max_abs_diff < TOLERANCE:
    print(f"\n  ✅  PASS — outputs agree within {TOLERANCE} (float32 tolerance)")
else:
    print(f"\n  ❌  FAIL — max diff {max_abs_diff:.2e} exceeds {TOLERANCE}")
    print("     Check the weight mapping above for errors.")

---
## Stage 12 — Stage-by-Stage Intermediate Shape Inspection

Run a single image through and print the tensor shape **after every stage**  
to confirm the data-flow matches the architecture table at the top.

In [ ]:
torch.manual_seed(0)
x = torch.randn(1, 3, 224, 224)

simple_model.eval()
with torch.no_grad():

    print(f"[Input]              x : {list(x.shape)}")

    # Stage 1 — patch embed
    x = simple_model.patch_embed(x)
    print(f"[Stage 1 PatchEmbed] x : {list(x.shape)}   ← (B, 196, 192)")

    # Stage 2 — CLS
    cls = simple_model.cls_token.expand(x.shape[0], -1, -1)
    x   = torch.cat([cls, x], dim=1)
    print(f"[Stage 2 CLS token]  x : {list(x.shape)}   ← (B, 197, 192)")

    # Stage 3 — pos embed
    x = x + simple_model.pos_embed
    print(f"[Stage 3 PosEmbed]   x : {list(x.shape)}   ← (B, 197, 192)")

    # Stage 5 — transformer blocks
    for i, blk in enumerate(simple_model.blocks):
        x, attn = blk(x)
        print(f"[Stage 5 Block {i:02d}]    x : {list(x.shape)}   attn: {list(attn.shape)}")

    # Stage 6 — final norm
    x_normed = simple_model.clf_head.norm(x)
    print(f"[Stage 6 FinalNorm]  x : {list(x_normed.shape)}")

    # Stage 7 — CLS extraction
    cls_out = x_normed[:, 0]
    print(f"[Stage 7 CLS out]    x : {list(cls_out.shape)}   ← (B, 192)")

    # Stage 8 — head
    logits = simple_model.clf_head.head(cls_out)
    print(f"[Stage 8 Head]       x : {list(logits.shape)}   ← (B, 2)")

    # Stage 9 — softmax
    probs = F.softmax(logits, dim=-1)
    print(f"[Stage 9 Softmax]    : benign={probs[0,0]:.4f}  malware={probs[0,1]:.4f}")
    print(f"                       → predicted: {'malware' if probs[0,1] > probs[0,0] else 'benign'}")

---
## Stage 13 — MLP Weight Cross-Check vs W1.mem / W2.mem

Verify that the MLP weights in `simple_model.blocks[0].mlp` match the Q8.8 fixed-point  
values stored in the hardware accelerator memory files.

In [ ]:
import numpy as np

SCALE = 256  # Q8.8

mlp0 = simple_model.blocks[0].mlp
W1 = mlp0.fc1.weight.detach().cpu()
W2 = mlp0.fc2.weight.detach().cpu()

W1_fixed = (W1 * SCALE).round().int()
W2_fixed = (W2 * SCALE).round().int()

print("MLP Block 0 — weight statistics")
print(f"  W1 (fc1) shape  : {list(W1.shape)}  (out=768, in=192)")
print(f"  W1 float range  : [{W1.min():.4f}, {W1.max():.4f}]")
print(f"  W1 Q8.8  range  : [{W1_fixed.min().item()}, {W1_fixed.max().item()}]")
print()
print(f"  W2 (fc2) shape  : {list(W2.shape)}  (out=192, in=768)")
print(f"  W2 float range  : [{W2.min():.4f}, {W2.max():.4f}]")
print(f"  W2 Q8.8  range  : [{W2_fixed.min().item()}, {W2_fixed.max().item()}]")

for fname, W_fixed, label in [
    ("W1.mem", W1_fixed, "fc1"),
    ("W2.mem", W2_fixed, "fc2")
]:
    if os.path.exists(fname):
        mem = np.loadtxt(fname, dtype=int)
        match = np.array_equal(mem, W_fixed.numpy().flatten())
        status = "✅ PASS" if match else "❌ MISMATCH"
        print(f"\n  {fname} ({label}) cross-check: {status}")
        if not match:
            diff = (mem - W_fixed.numpy().flatten())
            print(f"  Max diff: {np.abs(diff).max()}")
    else:
        print(f"\n  {fname}: not found — skipping")

---
## Stage 14 — Attention Map Visualisation  

Inspect what each attention head focuses on for the CLS token across all 12 blocks.

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(42)
test_img = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    _, all_attn = simple_model(test_img, return_attn=True)

# all_attn: list of 12 tensors, each (1, 3_heads, 197, 197)
# We look at CLS token row (index 0) → how it attends to patches (indices 1..196)

fig, axes = plt.subplots(4, 3, figsize=(14, 18))
axes = axes.flatten()

for blk_idx, attn in enumerate(all_attn):
    # Average over 3 heads, then take CLS row
    cls_attn = attn[0].mean(dim=0)[0, 1:]   # (196,) — CLS→patches
    cls_attn_map = cls_attn.reshape(14, 14).numpy()

    ax = axes[blk_idx]
    im = ax.imshow(cls_attn_map, cmap="inferno", interpolation="nearest")
    ax.set_title(f"Block {blk_idx:02d} — CLS attention (avg 3 heads)", fontsize=9)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("CLS Token Attention Maps (14×14 patch grid) — All 12 Blocks", fontsize=13, fontweight="bold")
plt.tight_layout()

os.makedirs("architecture_graphs", exist_ok=True)
plt.savefig("architecture_graphs/attention_maps.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → architecture_graphs/attention_maps.png")

---
## Stage 15 — Per-Block Intermediate Tensor Stats

For RTL planning: measure the **dynamic range** (min/max/std) of activations  
at each key point to guide fixed-point bit-width selection.

In [ ]:
print(f"{'Stage':<35} {'min':>9} {'max':>9} {'std':>9} {'shape'}")
print("-" * 80)

torch.manual_seed(0)
x = torch.randn(1, 3, 224, 224)

def stats(name, t):
    print(f"  {name:<33} {t.min().item():+9.4f} {t.max().item():+9.4f} {t.std().item():9.4f}  {list(t.shape)}")

simple_model.eval()
with torch.no_grad():
    stats("Input image", x)

    x = simple_model.patch_embed(x)
    stats("After PatchEmbed", x)

    cls = simple_model.cls_token.expand(1, -1, -1)
    x   = torch.cat([cls, x], dim=1)
    x   = x + simple_model.pos_embed
    stats("After PosEmbed", x)

    for i, blk in enumerate(simple_model.blocks):
        # Mid-block: after norm1 but before attention
        x_n1 = blk.norm1(x)
        stats(f"Block {i:02d} → after LN1", x_n1)

        attn_out, _ = blk.attn(x_n1)
        stats(f"Block {i:02d} → after MHSA", attn_out)

        x = x + attn_out
        x_n2 = blk.norm2(x)
        mlp_out = blk.mlp(x_n2)
        stats(f"Block {i:02d} → after MLP", mlp_out)

        x = x + mlp_out
        stats(f"Block {i:02d} → output", x)

    x_normed = simple_model.clf_head.norm(x)
    stats("After final LN", x_normed)

    cls_out = x_normed[:, 0]
    stats("CLS token", cls_out)

    logits = simple_model.clf_head.head(cls_out)
    stats("Logits", logits)

---
## Summary

| Stage | Class | Key operation | Param count |
|---|---|---|---|
| 1 | `SimplePatchEmbed` | Conv2d(3, 192, 16, stride=16) | 147,648 |
| 2–3 | `SimpleViT` (params) | CLS token + PosEmbed | 38,016 |
| 4–5 (×12) | `SimpleTransformerBlock` | LN → MHSA → LN → MLP | ~443K each |
| 6–8 | `SimpleClassificationHead` | LN + Linear(192,2) | 386 |

**Next steps:**
1. Verify Stage 11 (numerical equivalence) passes ✅
2. Verify Stage 13 (W1.mem/W2.mem cross-check) passes ✅
3. Study dynamic ranges in Stage 15 to choose fixed-point formats per stage
4. Begin RTL: Patch Embedding → MLP block → Attention → Final Head